In [7]:
# basic libraries
import numpy as np
import xarray as xr
import pandas as pd
import matplotlib.pyplot as plt
from pandas.plotting import register_matplotlib_converters

# libraries for maps
import cartopy.crs as ccrs
import cartopy.feature as cfeature

# units libraries
import metpy.calc as mpcalc
from metpy.units import units

# time libraries
import matplotlib.dates as mdates

# Plots libraries
import matplotlib.colors as mcolors
import seaborn as sns
import matplotlib.colors as colors
from matplotlib.lines import Line2D
import os

# buoy analysis library
from scipy.spatial import cKDTree

Donwload buoy dataset

In [ ]:
import copernicusmarine

copernicusmarine.subset(
  dataset_id="cmems_obs-ins_med_phybgcwav_mynrt_na_irr",
  dataset_part="monthly",
  variables=["VAVH", "VGHS", "VHM0"],
  minimum_longitude=-6,
  maximum_longitude=37,
  minimum_latitude=30,
  maximum_latitude=46,
  start_datetime="2020-01-01T00:00:00",
  end_datetime="2021-01-01T00:00:00",
  minimum_depth=0,
  maximum_depth=0,
  output_filename="C://Users//kosta//OneDrive//Desktop//Διπλωματική//Data//byos"  # where to store the data
)

""" Some other dataset
copernicusmarine.subset(
  dataset_id="cmems_obs-wind_med_phy_my_l3-s1a-sar-desc-0.01deg_P1D-i",
  variables=["eastward_wind", "northward_wind", "rejection_flag", "measurement_time", "quality_level"],
  minimum_longitude=-5.604000091552734,
  maximum_longitude=36.80400085449219,
  minimum_latitude=30.125999450683594,
  maximum_latitude=46.00199890136719,
  start_datetime="2020-09-19T00:00:00",
  end_datetime="2020-09-20T00:00:00",
  output_filename="C://Users//kosta//OneDrive//Desktop//Διπλωματική//Data"  # where to store the data
)"""

INFO - 2025-11-27T10:12:18Z - Downloading Copernicus Marine data requires a Copernicus Marine username and password, sign up for free at: https://data.marine.copernicus.eu/register


Copernicus Marine username:Copernicus Marine password:

INFO - 2025-11-27T10:12:27Z - Selected dataset version: "202311"
INFO - 2025-11-27T10:12:27Z - Selected dataset part: "monthly"


ResponseSubset(file_path=WindowsPath('C:/Users/kosta/OneDrive/Desktop/Διπλωματική/Data/byos.csv'), output_directory=WindowsPath('.'), filename='C:\\Users\\kosta\\OneDrive\\Desktop\\Διπλωματική\\Data\\byos.csv', file_size=None, data_transfer_size=None, variables=['VAVH', 'VGHS', 'VHM0'], coordinates_extent=[], status='000', message='The request was successful.', file_status='DOWNLOADED')

Dataset used for latitude, longitude definition

In [ ]:
ds_0 = xr.open_dataset("C://Users//kosta//OneDrive//Desktop//Διπλωματική//Results//Maps_diff//hs_mean_diff01.nc", engine="h5netcdf")
ds_0


<xarray.Dataset> Size: 16MB
Dimensions:    (time: 1, bnds: 2, latitude: 787, longitude: 1651)
Coordinates:
    latitude   (latitude, longitude) float32 5MB ...
    longitude  (latitude, longitude) float32 5MB ...
  * time       (time) datetime64[ns] 8B 2020-01-16T11:30:00
Dimensions without coordinates: bnds
Data variables:
    time_bnds  (time, bnds) datetime64[ns] 16B ...
    diff       (time, latitude, longitude) float32 5MB ...
Attributes: (12/22)
    CDI:                              Climate Data Interface version 1.9.6 (h...
    Conventions:                      CF-1.6
    WAVEWATCH_III_version_number:     6.07
    WAVEWATCH_III_switches:           F90 NOGRB NC4 SHRD PR3 UQ FLX0 LN1 ST4 ...
    SIN4 namelist parameter BETAMAX:  1.55
    product_name:                     ww3.run_with_currents202001.nc
    ...                               ...
    start_date:                       2020-01-01 00:00:00
    stop_date:                        2020-01-31 23:00:00
    CDO:                              Climate Data Operators version 1.9.6 (h...
    history:                          Thu Oct  9 13:26:05 2025: ncks -A lan_l...
    history_of_appended_files:        Thu Oct  9 13:26:05 2025: Appended file...
    NCO:                              "4.5.2"

Buoys dataset

In [2]:
import pandas as pd
csv_file = "C://Users//kosta//OneDrive//Desktop//Διπλωματική//Data//byos.csv"

# 1. Read the CSV into a pandas DataFrame
df_altim = pd.read_csv(csv_file)
print("Pandas DataFrame:")
print(df_altim)

Pandas DataFrame:
       variable           platform_id platform_type                  time  \
0          VAVH               1801573            SD  2020-03-05T22:30:00Z   
1          VAVH               1801573            SD  2020-03-05T23:00:00Z   
2          VAVH               1801573            SD  2020-03-05T23:30:00Z   
3          VAVH               1801573            SD  2020-03-06T00:00:00Z   
4          VAVH               1801573            SD  2020-03-06T00:30:00Z   
...         ...                   ...           ...                   ...   
546185     VHM0  Tarragona-coast-buoy            MO  2020-12-31T20:00:00Z   
546186     VHM0  Tarragona-coast-buoy            MO  2020-12-31T21:00:00Z   
546187     VHM0  Tarragona-coast-buoy            MO  2020-12-31T22:00:00Z   
546188     VHM0  Tarragona-coast-buoy            MO  2020-12-31T23:00:00Z   
546189     VHM0  Tarragona-coast-buoy            MO  2021-01-01T00:00:00Z   

        longitude  latitude  depth  pressure  is_depth_fr

C:\Users\kosta\AppData\Local\Temp\ipykernel_6608\2298872534.py:5: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  df_altim = pd.read_csv(csv_file)


Drop nan values of a parameter

In [4]:
df_altim = df_altim.dropna(subset=['value'])
df_altim

,variable,platform_id,platform_type,time,longitude,latitude,depth,pressure,is_depth_from_producer,value,value_qc,institution,doi,product_doi
0,VAVH,1801573,SD,2020-03-05T22:30:00Z,-5.99110,36.01513,-0.0,NaN,1,1.230,1,Saildrone,https://doi.org/10.13155/59938 https://doi.org...,https://doi.org/10.48670/moi-00044
1,VAVH,1801573,SD,2020-03-05T23:00:00Z,-5.94570,36.01550,-0.0,NaN,1,1.202,1,Saildrone,https://doi.org/10.13155/59938 https://doi.org...,https://doi.org/10.48670/moi-00044
2,VAVH,1801573,SD,2020-03-05T23:30:00Z,-5.90193,36.00592,-0.0,NaN,1,1.252,1,Saildrone,https://doi.org/10.13155/59938 https://doi.org...,https://doi.org/10.48670/moi-00044
3,VAVH,1801573,SD,2020-03-06T00:00:00Z,-5.86360,36.01642,-0.0,NaN,1,1.395,1,Saildrone,https://doi.org/10.13155/59938 https://doi.org...,https://doi.org/10.48670/moi-00044
4,VAVH,1801573,SD,2020-03-06T00:30:00Z,-5.88002,36.00364,-0.0,NaN,1,1.341,1,Saildrone,https://doi.org/10.13155/59938 https://doi.org...,https://doi.org/10.48670/moi-00044
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
546185,VHM0,Tarragona-coast-buoy,MO,2020-12-31T20:00:00Z,1.19000,41.07000,-0.0,NaN,1,0.380,1,Puertos del Estado,https://doi.org/10.13155/59938 https://doi.org...,https://doi.org/10.48670/moi-00044
546186,VHM0,Tarragona-coast-buoy,MO,2020-12-31T21:00:00Z,1.19000,41.07000,-0.0,NaN,1,0.360,1,Puertos del Estado,https://doi.org/10.13155/59938 https://doi.org...,https://doi.org/10.48670/moi-00044
546187,VHM0,Tarragona-coast-buoy,MO,2020-12-31T22:00:00Z,1.19000,41.07000,-0.0,NaN,1,0.380,1,Puertos del Estado,https://doi.org/10.13155/59938 https://doi.org...,https://doi.org/10.48670/moi-00044
546188,VHM0,Tarragona-coast-buoy,MO,2020-12-31T23:00:00Z,1.19000,41.07000,-0.0,NaN,1,0.460,1,Puertos del Estado,https://doi.org/10.13155/59938 https://doi.org...,https://doi.org/10.48670/moi-00044


Check platform's id

In [ ]:
# using drop_duplicates() to drop duplicate values, and reset_index() to reset the index of the dataset, in order to know how many buoys we have.
# Finally, we used drop=True, inside reset_index() to drop a new column, named "index", that would be added via reset_index() fuction.
unique_platform = df_altim[['platform_id']].drop_duplicates().reset_index(drop=True)
unique_platform

,platform_id
0,1801573
1,5801955
2,6100002
3,6100021
4,6100022
5,6100188
6,6100190
7,6100191
8,6100289
9,6100294


Check data by each platform

In [43]:
df_HERAKLION = df_altim[df_altim['platform_id'] == 'HERAKLION']
#df_HERAKLION = df_HERAKLION[['value']] # if you want to check only the SWH measured by the buoy in HERAKLION station
df_HERAKLION

,variable,platform_id,platform_type,time,longitude,latitude,depth,pressure,is_depth_from_producer,value,value_qc,institution,doi,product_doi
515810,VHM0,HERAKLION,MO,2020-01-06T18:00:00Z,25.0792,35.4342,-0.0,NaN,1,4.570,1,Hellenic Centre for Marine Research - Institut...,https://doi.org/10.48670/moi-00044 https://doi...,https://doi.org/10.48670/moi-00044
515811,VHM0,HERAKLION,MO,2020-01-06T21:00:00Z,25.0792,35.4342,-0.0,NaN,1,4.453,1,Hellenic Centre for Marine Research - Institut...,https://doi.org/10.48670/moi-00044 https://doi...,https://doi.org/10.48670/moi-00044
515812,VHM0,HERAKLION,MO,2020-01-07T00:00:00Z,25.0792,35.4342,-0.0,NaN,1,5.273,1,Hellenic Centre for Marine Research - Institut...,https://doi.org/10.48670/moi-00044 https://doi...,https://doi.org/10.48670/moi-00044
515813,VHM0,HERAKLION,MO,2020-01-07T06:00:00Z,25.0792,35.4342,-0.0,NaN,1,5.156,1,Hellenic Centre for Marine Research - Institut...,https://doi.org/10.48670/moi-00044 https://doi...,https://doi.org/10.48670/moi-00044
515814,VHM0,HERAKLION,MO,2020-01-07T09:00:00Z,25.0792,35.4342,-0.0,NaN,1,4.687,1,Hellenic Centre for Marine Research - Institut...,https://doi.org/10.48670/moi-00044 https://doi...,https://doi.org/10.48670/moi-00044
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
517605,VHM0,HERAKLION,MO,2020-12-30T18:00:00Z,25.0792,35.4342,-0.0,NaN,1,0.505,1,Hellenic Centre for Marine Research - Institut...,https://doi.org/10.48670/moi-00044 https://doi...,https://doi.org/10.48670/moi-00044
517606,VHM0,HERAKLION,MO,2020-12-30T21:00:00Z,25.0792,35.4342,-0.0,NaN,1,0.374,1,Hellenic Centre for Marine Research - Institut...,https://doi.org/10.48670/moi-00044 https://doi...,https://doi.org/10.48670/moi-00044
517607,VHM0,HERAKLION,MO,2020-12-31T18:00:00Z,25.0792,35.4342,-0.0,NaN,1,0.000,1,Hellenic Centre for Marine Research - Institut...,https://doi.org/10.48670/moi-00044 https://doi...,https://doi.org/10.48670/moi-00044
517608,VHM0,HERAKLION,MO,2020-12-31T21:00:00Z,25.0792,35.4342,-0.0,NaN,1,0.346,1,Hellenic Centre for Marine Research - Institut...,https://doi.org/10.48670/moi-00044 https://doi...,https://doi.org/10.48670/moi-00044


Find the unique latitude/longitude of each measurement

In [8]:
unique_coords = df_altim[['longitude', 'latitude']].drop_duplicates()
unique_coords

,longitude,latitude
0,-5.99110,36.01513
1,-5.94570,36.01550
2,-5.90193,36.00592
3,-5.86360,36.01642
4,-5.88002,36.00364
...,...,...
459616,2.70040,39.49280
465294,21.60680,36.82880
497254,24.72940,39.97500
515810,25.07920,35.43420


Find the range of lon/lat of measurements

In [9]:
lon_min, lon_max = df_altim['longitude'].min(), df_altim['longitude'].max()
lat_min, lat_max = df_altim['latitude'].min(), df_altim['latitude'].max()

print("Longitude range:", lon_min, lon_max)
print("Latitude range:", lat_min, lat_max)

Longitude range: -5.9911 25.4597
Latitude range: 35.327 45.6995


Counts the number of measurements in each lon/lat

In [10]:
lon_counts = df_altim['longitude'].value_counts()
lat_counts = df_altim['latitude'].value_counts()
lon_counts, lat_counts

(longitude
  3.77970    34792
  3.12450    34377
  5.23000    34356
  4.13330    32069
  3.16830    31945
             ...  
 -4.22956        1
 -4.22845        1
 -4.22876        1
 -4.22914        1
 -5.86870        1
 Name: count, Length: 11066, dtype: int64,
 latitude
 43.37150    34792
 42.91620    34377
 43.20834    34356
 43.42500    32069
 42.48830    31945
             ...  
 36.28476        1
 36.28465        1
 36.28501        1
 36.28559        1
 38.08967        1
 Name: count, Length: 10977, dtype: int64)

Check the unique dates of measurements

In [12]:
df_altim['time'] = pd.to_datetime(df_altim['time'])

# Extract only the date part (YYYY-MM-DD)
df_altim['date'] = df_altim['time'].dt.date

# Get unique dates
unique_dates = df_altim['date'].unique()
unique_dates

array([datetime.date(2020, 3, 5), datetime.date(2020, 3, 6),
       datetime.date(2020, 3, 7), datetime.date(2020, 3, 8),
       datetime.date(2020, 3, 9), datetime.date(2020, 3, 10),
       datetime.date(2020, 3, 11), datetime.date(2020, 3, 12),
       datetime.date(2020, 3, 13), datetime.date(2020, 3, 14),
       datetime.date(2020, 3, 15), datetime.date(2020, 3, 16),
       datetime.date(2020, 3, 17), datetime.date(2020, 3, 18),
       datetime.date(2020, 3, 19), datetime.date(2020, 3, 20),
       datetime.date(2020, 3, 21), datetime.date(2020, 3, 22),
       datetime.date(2020, 3, 23), datetime.date(2020, 3, 24),
       datetime.date(2020, 3, 25), datetime.date(2020, 3, 26),
       datetime.date(2020, 3, 27), datetime.date(2020, 3, 28),
       datetime.date(2020, 3, 29), datetime.date(2020, 3, 30),
       datetime.date(2020, 3, 31), datetime.date(2020, 4, 1),
       datetime.date(2020, 4, 2), datetime.date(2020, 4, 3),
       datetime.date(2020, 4, 4), datetime.date(2020, 4, 5),
  

Check the lon/lat of each buoy platform

In [ ]:
# group dataset by the platform id and show lat and lon in each platform, while also reset the index of dataset
lat_lon_list = df_altim.groupby('platform_id')[['latitude', 'longitude']].first().reset_index()
lat_lon_list


,platform_id,latitude,longitude
0,61277,35.726300,25.130700
1,1801573,36.015130,-5.991100
2,5801955,36.005120,-5.983470
3,6100002,42.071800,4.650300
4,6100021,42.930000,6.207000
5,6100022,43.713580,7.426170
6,6100188,42.488300,3.168300
7,6100190,43.371500,3.779700
8,6100191,42.916200,3.124500
9,6100196,41.900000,3.650000


Find the corresponding indecies of each platfrom on model's domain via a kdtree

In [ ]:
# require libraries
import numpy as np
import pandas as pd
from scipy.spatial import cKDTree

# -------------------------------------------
# 1. PREPARE MODEL GRID & KDTREE (done once)
# -------------------------------------------
lat2d = ds_0["latitude"].values
lon2d = ds_0["longitude"].values

ny, nx = lat2d.shape
lat_flat = lat2d.ravel()
lon_flat = lon2d.ravel()

coords_model = np.column_stack([
    np.radians(lat_flat),
    np.radians(lon_flat)
])
tree = cKDTree(coords_model)

# -------------------------------------------
# 2. LOOP OVER PLATFORMS
# -------------------------------------------
results = []

for platform, df_plat in df_altim.groupby("platform_id"):

    lat_obs = df_plat["latitude"].values
    lon_obs = df_plat["longitude"].values

    coords_obs = np.column_stack([
        np.radians(lat_obs),
        np.radians(lon_obs)
    ])
    
    # Spatial match
    dist, idx_flat = tree.query(coords_obs)
    j_idx, i_idx = np.unravel_index(idx_flat, (ny, nx))

    # Build DataFrame for this platform
    df_out = pd.DataFrame({
        "platform": platform,
        "latitude": lat_obs,
        "longitude": lon_obs,
        "j_idx": j_idx,
        "i_idx": i_idx
    })

    results.append(df_out)

# -------------------------------------------
# 3. CONCAT RESULTS FOR ALL PLATFORMS
# -------------------------------------------
df_kdtree_output = pd.concat(results, ignore_index=True)
df_kdtree_output


,platform,latitude,longitude,j_idx,i_idx
0,61277,35.7263,25.1307,259,1249
1,61277,35.7263,25.1307,259,1249
2,61277,35.7263,25.1307,259,1249
3,61277,35.7263,25.1307,259,1249
4,61277,35.7263,25.1307,259,1249
...,...,...,...,...,...
546185,Tarragona-coast-buoy,41.0700,1.1900,493,381
546186,Tarragona-coast-buoy,41.0700,1.1900,493,381
546187,Tarragona-coast-buoy,41.0700,1.1900,493,381
546188,Tarragona-coast-buoy,41.0700,1.1900,493,381


In [14]:
index_list = df_kdtree_output.groupby('platform')[['latitude', 'longitude', 
                                                   'j_idx', 'i_idx']].first().reset_index()
index_list

,platform,latitude,longitude,j_idx,i_idx
0,61277,35.726300,25.130700,259,1249
1,1801573,36.015130,-5.991100,248,130
2,5801955,36.005120,-5.983470,248,130
3,6100002,42.071800,4.650300,549,502
4,6100021,42.930000,6.207000,597,555
5,6100022,43.713580,7.426170,641,596
6,6100188,42.488300,3.168300,567,449
7,6100190,43.371500,3.779700,614,468
8,6100191,42.916200,3.124500,589,446
9,6100196,41.900000,3.650000,538,467
